# Phase F1 — full fine-tuning on Kaggle

This notebook launches the private F1 runner. It starts from the original pinned `neurlang/ipa-whisper-base` checkpoint, uses all 3,629 frozen training clips, validates on all 732 validation clips, and never reads the sealed test split.

Attach three private inputs: (1) the full training bundle with `bundle_type=phase7_f1_train_v1`, (2) the Phase 5 validation bundle with `bundle_type=phase5_validation_baseline`, and (3) `scripts/phase7_f1_train.py`. Select one NVIDIA T4 GPU and enable Internet only to fetch the pinned public model if it is not cached.

In [ ]:
import importlib.metadata as md, subprocess, sys
PINS={'transformers':'4.57.1','tokenizers':'0.22.2','huggingface-hub':'0.36.2','safetensors':'0.8.0'}
installed={}
for p in PINS:
    try: installed[p]=md.version(p)
    except md.PackageNotFoundError: installed[p]=None
print('Installed:',installed)
if installed!=PINS:
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--no-deps',*[f'{p}=={v}' for p,v in PINS.items()]],check=True)
    raise RuntimeError('Packages were pinned. Restart Kaggle and run all cells again.')
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA GPU'
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import subprocess,sys
root=Path('/kaggle/input'); candidates=list(root.rglob('phase7_f1_train.py'))
if not candidates: raise FileNotFoundError('Attach scripts/phase7_f1_train.py as a private Kaggle input.')
runner=candidates[0]; out=Path('/kaggle/working/phase7_F1_private')
cmd=[sys.executable,str(runner),'--output',str(out),'--physical-batch','2','--accumulation','4','--validation-batch','8','--max-epochs','10','--patience','3','--learning-rate','1e-5','--weight-decay','0.01']
print('Running:', ' '.join(cmd))
print('F1 uses lazy CPU feature extraction, FP16 AMP training, batch-8 FP32 validation, resumable epoch checkpoints, and tqdm progress bars.')
subprocess.run(cmd,check=True)

## F1 safety gates

The runner stops if the GPU is unavailable, the model revision or bundle counts are wrong, the validation denominator is not 64,137 code points, an input is non-finite, gradients are non-finite, or a checkpoint fingerprint does not match on resume.

If a CUDA out-of-memory error occurs, restart from a clean session and rerun with `--physical-batch 1 --accumulation 8`; this preserves effective batch size 8. Do not change the frozen split, normalization, CER implementation, model revision, or test-set seal.

In [ ]:
import json
report=json.loads((out/'F1_report_private.json').read_text())
print({k:report[k] for k in ['best_epoch','best_validation_cer','completed_steps','total_wall_seconds','peak_allocated_mib','test_data_used']})
assert report['status']=='complete' and report['test_data_used'] is False
print('Private outputs:',out)